# Label Invasion using Microsam Outputs and Edge Detection

## CUDA Test

In [ ]:
# import torch

# print(f"PyTorch version: {torch.__version__}")
# print(f"CUDA available: {torch.cuda.is_available()}")
# print(f"Device count: {torch.cuda.device_count()}")

# if torch.cuda.is_available():
#     try:
#         print(f"Current device: {torch.cuda.current_device()}")
#         print(f"Device name: {torch.cuda.get_device_name(0)}")
#         # This specific call often triggers the NVML error
#         print(f"Memory Usage: {torch.cuda.memory_allocated()}")
#     except Exception as e:
#         print(f"Error accessing device details: {e}")

## Load Files

In [ ]:
from image_processing_tools.util.load_files import find_files_by_pattern
from image_processing_tools.image_class.image_container import ImageContainer
from pathlib import Path
import matplotlib
# matplotlib.use('Agg') # Critical: prevents plotting to screen/RAM

search_path = (
    "~/A8/Data_Roan/260123_Cocu_Phalloidin/Cocu_Cet112_Ca922_Phalloidin_01_CY5, DAPI",
    "~/A8/Data_Roan/260123_Cocu_Phalloidin/Cocu_Cet112_Ca922_Phalloidin_01_CY5, DAPI, BF",
    "~/A8/Data_Roan/260123_Cocu_Phalloidin/Cocu_Cet112_Ca922_Phalloidin_02_CY5, DAPI",
    "~/A8/Data_Roan/260123_Cocu_Phalloidin/Cocu_Cet112_Ca922_Phalloidin_low_01_CY5, DAPI",
    "~/A8/Data_Roan/260123_Cocu_Phalloidin/Cocu_Cet112_Ca922_Phalloidin_low_01_DIC",
    "~/A8/Data_Roan/260123_Cocu_Phalloidin/Cocu_Cet112_Ca922_Phalloidin_low_01_FITC, BF",
    )

file_pattern = (
    "*MAX*.tif",
    "*SHARPEST-normalized_variance*.tif",
    "*SHARPEST-manual*.tif",
    "SUBTRACT-direct*O5*.tif"
    )

config = {
    "preprocessing": {
        "resize_image": False,
        "max_dim": 1080,
        "outlier_percentile": 0.35,
        "quantization": "16bit"
    }
}

config_DIC = {
    "preprocessing": {
        "resize_image": True,
        "max_dim": 1080,
        "outlier_percentile": 0.35,
        "quantization": "8bit",
        "correct_DIC_shift": True
    }
}

# Find the files. The 'files' variable will hold the list of Path objects.
files = find_files_by_pattern(search_path[3], file_pattern[0], verbose=True)

# Create output directory
base_input_dir = Path("~/A8/Data_Roan/").expanduser()
base_output_dir = Path("output/label_invasion_microsam")

# Assuming 'files' is defined in your notebook environment as a list of Path objects
relative_dir = files[0].parent.relative_to(base_input_dir)
current_output_dir = base_output_dir / relative_dir
current_output_dir.mkdir(parents=True, exist_ok=True)

print(f"The current output dir is {current_output_dir}")

In [ ]:
cy5 = ImageContainer(files[0],config)
cy5_img = cy5.merge()

## Edge Detection

First, we run edge detection using Sobel vs. Frangi filter for a comparison

Note: Although difficult to get the cell outline, Sobel can still be used to get actin fibers or summed with Frangi to get cell outline (maybe). TBD later

### Image Pre-processing

In [ ]:
from image_processing_tools.image_class.image_filters import ImageJProcessor
import matplotlib.pyplot as plt

cy5_frangi = ImageJProcessor(cy5_img)
cy5_frangi.reset()
cy5_frangi.enhance_contrast_clahe(slope=4)
cy5_frangi.gaussian_blur(sigma=5)
cy5_frangi.fft_bandpass_filter(25,2)
cy5_frangi.imagej_rolling_ball(radius=60, use_paraboloid=False)
cy5_processed = cy5_frangi.return_image()

fig, axes = plt.subplots(1,2,figsize=(6,12))
axes[0].imshow(cy5_img, cmap='gray'); axes[0].set_title('Original'); axes[0].axis('off')
axes[1].imshow(cy5_processed, cmap='gray'); axes[1].set_title('Processed'); axes[1].axis('off')

plt.tight_layout()
plt.show()

### Sobel Filter

In [ ]:
from image_processing_tools.util.phalloidin_processing import process_actin_fibers
import gc
import matplotlib.pyplot as plt
import numpy as np
from image_processing_tools.util.phalloidin_processing import get_otsu_edge_map
# from scipy.ndimage import binary_dilation

arrow_params = {
    'flip_opposing_vectors': False,
    'use_log_magnitude': True,
    'clip_magnitude': True,
}

cy5_edges = process_actin_fibers(cy5_processed, edge_method = "sobel", **arrow_params)

def get_mag(edges):
    gx, gy = edges
    mag = np.sqrt(gx**2 + gy**2)
    return mag

cy5_mag = get_mag(cy5_edges)
cy5_mag = get_otsu_edge_map(cy5_mag)

### Frangi Filter

In [ ]:
vesselness, scalemap = cy5_frangi.frangi_imagej_ops(scales=[6,8],beta=1,c=15)

vesselness = get_otsu_edge_map(vesselness)

fig, axes = plt.subplots(1,2,figsize=(6,12))
axes[0].imshow(cy5_mag, cmap='viridis'); axes[0].set_title('Sobel'); axes[0].axis('off')
axes[1].imshow(vesselness, cmap='viridis'); axes[1].set_title('Frangi'); axes[1].axis('off')

plt.tight_layout()
plt.show()

## Microsam Segmentation

In [ ]:
from image_processing_tools.image_class.micro_sam_pipeline import MicroSAMPipeline, load_config
from pathlib import Path

config = load_config("micro_sam_config.json")

config["segmentation_mode"] = "prompted"
config["prompting"]["prompt_mode"] = "points" # bbox or points, mask in implementation process
config["prompting"]["bbox_radius_multiplier"] = 3.5

config['preprocessing']['resize_image'] = False

config['base_input_dir'] = Path("~/A8/Data_Roan/").expanduser()
config['checkpoint_path'] = Path("~/Projects/C_Albicans Thesis Project/7. Data/model_checkpoints/micro_sam/models/vit_l_lm").expanduser()
config['decoder_checkpoint_path'] = Path("~/Projects/C_Albicans Thesis Project/7. Data/model_checkpoints/micro_sam/models/vit_l_lm_decoder").expanduser()
config['model_type'] = 'vit_l_lm'

microsam_pipe = MicroSAMPipeline(search_path[3], file_pattern[0], config)

In [ ]:
# 6. Define your analysis plan using the indices printed above.
# microsam takes 3 channels
analysis_plan = [
    [0,0,2],
]

# 7. Execute the segmentation runs defined in your analysis plan.
# The pipeline will use the DAPI channel at the specified index to generate prompts
# and will cache image data to avoid reloading files used in multiple runs.
# logging.info("Starting segmentation runs...")
microsam_pipe.run(run_spec=analysis_plan)
# logging.info("All segmentation runs completed.")

# 8. Visualize the results.
# This will save a separate visualization for each run defined in the analysis_plan.
# Composite runs will be saved to a folder ending in '_comp'.
# logging.info("Generating visualizations...")
microsam_pipe.visualize_results(visualization_mode='channel_comparison') # or single
# logging.info("Visualizations have been saved to the 'output' directory.")

In [ ]:
res = microsam_pipe.results
res = res['MAX_Cocu_Cet112_Ca922_Phalloidin_low_01_CY5, DAPI_CY5,CY5,DAPI']
res_masks = res['masks']
res_img = res['processed_image']

In [ ]:
import numpy as np
from scipy.ndimage import binary_erosion
from typing import List, Union

def shrink_masks(masks: Union[List[np.ndarray], np.ndarray], iterations: int = 1) -> Union[List[np.ndarray], np.ndarray]:
    """
    Shrinks segmentation masks by eroding the boundary.

    Args:
        masks: The masks output from the pipeline.
               - List of arrays (Prompted mode)
               - Single label matrix (Automatic mode)
        iterations: Number of pixels to erode (shave off) from the edge. 
                    Higher numbers make the mask smaller.

    Returns:
        The shrunk masks in the same format as the input.
    """
    # Case A: Prompted Segmentation (List of masks)
    if isinstance(masks, list):
        shrunk_list = []
        for m in masks:
            # 1. Handle dimensions: (1, H, W) vs (H, W)
            is_3d = (m.ndim == 3)
            mask_2d = m[0] if is_3d else m
            
            # 2. Perform Erosion
            # border_value=0 ensures we don't erode from the image edge inwards if touching border
            eroded = binary_erosion(mask_2d, iterations=iterations, border_value=0)
            
            # 3. Restore dimensions
            if is_3d:
                shrunk_list.append(eroded[None, :, :])
            else:
                shrunk_list.append(eroded)
        return shrunk_list

    # Case B: Automatic Segmentation (Single Label Matrix)
    elif isinstance(masks, np.ndarray):
        shrunk_labels = np.zeros_like(masks)
        unique_ids = np.unique(masks)
        
        for uid in unique_ids:
            if uid == 0: continue # Skip background
            
            # Create binary mask for this specific object
            binary_mask = (masks == uid)
            
            # Erode
            eroded = binary_erosion(binary_mask, iterations=iterations, border_value=0)
            
            # Assign back to the new label matrix
            shrunk_labels[eroded] = uid
            
        return shrunk_labels
    
    return masks

In [ ]:
import numpy as np
from typing import List, Union, Optional

def extract_objects(image: np.ndarray, masks: Union[List[np.ndarray], np.ndarray]) -> List[np.ndarray]:
    """
    Extracts individual objects from an image based on segmentation masks.
    Each object is cropped to its bounding box, and the background is set to 0.

    Args:
        image (np.ndarray): The original image array (H, W) or (H, W, C).
        masks (Union[List[np.ndarray], np.ndarray]): 
            - A list of masks (from prompted segmentation), usually (1, H, W) or (H, W).
            - A single label matrix (from automatic segmentation), (H, W).

    Returns:
        List[np.ndarray]: A list of arrays, each containing one isolated object.
    """
    extracted_objects = []

    def _crop_and_mask(binary_mask: np.ndarray) -> Optional[np.ndarray]:
        """Helper to crop image to mask bounding box and zero out background."""
        # 1. Find bounding box indices
        rows = np.any(binary_mask, axis=1)
        cols = np.any(binary_mask, axis=0)
        
        if not np.any(rows) or not np.any(cols):
            return None # Handle empty masks
            
        rmin, rmax = np.where(rows)[0][[0, -1]]
        cmin, cmax = np.where(cols)[0][[0, -1]]

        # 2. Crop image and mask to the bounding box
        # Handle 3D (RGB) vs 2D (Grayscale) images
        if image.ndim == 3:
            crop_img = image[rmin:rmax+1, cmin:cmax+1, :]
            # Add channel dim to mask for broadcasting
            crop_mask = binary_mask[rmin:rmax+1, cmin:cmax+1, None]
        else:
            crop_img = image[rmin:rmax+1, cmin:cmax+1]
            crop_mask = binary_mask[rmin:rmax+1, cmin:cmax+1]
        
        # 3. Apply mask (set background to 0)
        return crop_img * crop_mask

    # --- Main Logic ---
    
    # Case A: Prompted Segmentation (List of masks)
    if isinstance(masks, list):
        for mask_arr in masks:
            # Normalize shape: Handle (1, H, W) vs (H, W)
            if mask_arr.ndim == 3 and mask_arr.shape[0] == 1:
                binary_mask = mask_arr[0] > 0
            else:
                binary_mask = mask_arr > 0
            
            obj = _crop_and_mask(binary_mask)
            if obj is not None:
                extracted_objects.append(obj)

    # Case B: Automatic Segmentation (Single Label Matrix)
    elif isinstance(masks, np.ndarray):
        unique_labels = np.unique(masks)
        # Iterate over all labels except 0 (background)
        for label in unique_labels:
            if label == 0:
                continue
            
            binary_mask = (masks == label)
            obj = _crop_and_mask(binary_mask)
            if obj is not None:
                extracted_objects.append(obj)

    return extracted_objects


shrinked_masks = shrink_masks(res_masks, iterations=10)
frangi_objects = extract_objects(vesselness,res_masks)
cell_objects = extract_objects(res_img,res_masks)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2,2,figsize=(10,6))
axes = axes.flatten()
axes[0].imshow(cell_objects[1], cmap='viridis'); axes[0].set_title('Non-invaded Cell'); axes[0].axis('off')
axes[1].imshow(frangi_objects[1], cmap='viridis'); axes[1].set_title('Frangi'); axes[1].axis('off')
axes[2].imshow(cell_objects[3], cmap='viridis'); axes[2].set_title('Invaded Cell'); axes[2].axis('off')
axes[3].imshow(frangi_objects[3], cmap='viridis'); axes[3].set_title('Frangi'); axes[3].axis('off')

plt.tight_layout()
plt.show()